In [1]:
#Step 1.1
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

df = pd.read_csv('amazon_labelled.txt', sep='\t')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/gabrielferraz/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/gabrielferraz/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gabrielferraz/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
#Step 1.2
s = df.loc[26, 'text']
print('Original Text')
print(s)

Original Text
I've owned this phone for 7 months now and can say that it's the best mobile phone I've had.


In [3]:
#Step 1.3
sTokenized = word_tokenize(s)
print('Tokenized with word_tokenize')
print(sTokenized)
print('\nOutput with generic split():')
print(s.split())

Tokenized with word_tokenize
['I', "'ve", 'owned', 'this', 'phone', 'for', '7', 'months', 'now', 'and', 'can', 'say', 'that', 'it', "'s", 'the', 'best', 'mobile', 'phone', 'I', "'ve", 'had', '.']

Output with generic split():
["I've", 'owned', 'this', 'phone', 'for', '7', 'months', 'now', 'and', 'can', 'say', 'that', "it's", 'the', 'best', 'mobile', 'phone', "I've", 'had.']


In [ ]:
#Question 1
#The standard s.split() splits on whitespace and attaches punctuation on specific words. word_tokenize will isolate contraction
#suffixes into separate tokens and detatches punctuation that is trailing.

In [4]:
#Step 1.4
def fix_contractions(token):
    if token == "'ve":
        return "have"
    elif token == "'s":
        return "is"
    return token

sTokenized = [fix_contractions(token) for token in sTokenized]
print('After Contraction Replacement')
print(sTokenized)

After Contraction Replacement
['I', 'have', 'owned', 'this', 'phone', 'for', '7', 'months', 'now', 'and', 'can', 'say', 'that', 'it', 'is', 'the', 'best', 'mobile', 'phone', 'I', 'have', 'had', '.']


In [5]:
#Step 1.5
sTokenized = [token for token in sTokenized if token.isalpha()]
print('After Removing Punctuation & Numbers')
print(sTokenized)

After Removing Punctuation & Numbers
['I', 'have', 'owned', 'this', 'phone', 'for', 'months', 'now', 'and', 'can', 'say', 'that', 'it', 'is', 'the', 'best', 'mobile', 'phone', 'I', 'have', 'had']


In [6]:
#Step 1.6
sTokenized = [token.lower() for token in sTokenized]
print('After lowercasing')
print(sTokenized)

After lowercasing
['i', 'have', 'owned', 'this', 'phone', 'for', 'months', 'now', 'and', 'can', 'say', 'that', 'it', 'is', 'the', 'best', 'mobile', 'phone', 'i', 'have', 'had']


In [7]:
# Step 1.7 
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))
sTokenized = [stemmer.stem(word) for word in sTokenized if word not in stop_words]

In [8]:
#Question 2
print('Final Form of sTokenized')
print(sTokenized)
print('Remaining tokens count:', len(sTokenized))

Final Form of sTokenized
['own', 'phone', 'month', 'say', 'best', 'mobil', 'phone']
Remaining tokens count: 7


In [9]:
#Step 2.1
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))
base_analyzer = CountVectorizer().build_analyzer()

def custom_preprocessor(doc):
    return [
        stemmer.stem(word) 
        for word in base_analyzer(doc) 
        if word not in stop_words and word.isalpha()]

vectorizer = CountVectorizer(analyzer=custom_preprocessor)
X = vectorizer.fit_transform(df['text'])
y = df['sentiment']

dfX = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print('Document-Term Matrix shape:', dfX.shape)

Document-Term Matrix shape: (1000, 1352)


In [11]:
#Step 2.2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42)

dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

nn_model = MLPClassifier(max_iter=1000)
nn_model.fit(X_train, y_train)
nn_pred = nn_model.predict(X_test)

def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    print(f"--- {name} Results ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}\n")
    return acc, prec, rec

In [12]:
#Question 3
evaluate_model('Decision Tree', y_test, dt_pred)
evaluate_model('Neural Network', y_test, nn_pred)

--- Decision Tree Results ---
Accuracy:  0.7833
Precision: 0.8069
Recall:    0.7597

--- Neural Network Results ---
Accuracy:  0.8000
Precision: 0.8133
Recall:    0.7922



(0.8, 0.8133333333333334, 0.7922077922077922)

In [13]:
#Step 3.1
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score

nltk.download('vader_lexicon')
sid = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/gabrielferraz/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [14]:
#Step 3.2
def get_positive_score(text):
    return sid.polarity_scores(text)['pos']

def get_negative_score(text):
    return sid.polarity_scores(text)['neg']

df['pos_score'] = df['text'].apply(get_positive_score)
df['neg_score'] = df['text'].apply(get_negative_score)

In [16]:
#Step 3.3
def classify_sentiment(row):
    if row['pos_score'] > row['neg_score']:
        return 1
    else:
        return 0

df['vader_pred'] = df.apply(classify_sentiment, axis=1)

vader_accuracy = accuracy_score(df['sentiment'], df['vader_pred'])
print('VADER Matching Percentage (Accuracy):', vader_accuracy * 100,'%')

VADER Matching Percentage (Accuracy): 83.2 %


In [72]:
#Question 4
#83% of the sentiment scores match. It seems like the VADER method was the ebst since it got a score of 83%. 
#The neural network mehtod was second with 8% accuracy. Then the decision tree method was last with 78%.